# Adaptive Learning Module — Phase 1
**Fixed concept set · simpleKT · EPPO with 3 difficulty levels**

This notebook trains and evaluates the basic adaptive module.
Expected runtime: **~15 minutes on Colab free T4**

What to look for in the results:
- APR should trend upward across episodes
- Goal% should increase over training
- EPPO should learn to target medium/hard for high-mastery concepts
- Steps taken should decrease as EPPO gets more efficient

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'scikit-learn', 'matplotlib', '-q'])
print('Ready')

In [ ]:
# Upload adaptive_module.py or paste it here
# If uploading via Colab files panel, skip the write cell below

# Check if file exists
import os
if os.path.exists('adaptive_module.py'):
    print('adaptive_module.py found')
else:
    print('Please upload adaptive_module.py to this runtime')

In [ ]:
from adaptive_module import (
    Config, SimpleKT, EPPOAgent, StudentState,
    SimulatedStudentEnv, train,
    evaluate_kt, evaluate_policy, run_full_evaluation, run_demo_session
)
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 4)

cfg = Config()
print(f'Device: {cfg.DEVICE}')
print(f'Concepts ({cfg.N_CONCEPTS}): {cfg.CONCEPTS}')
print(f'Action space: {cfg.N_ACTIONS} ({cfg.N_CONCEPTS} concepts x {cfg.DIFF_LEVELS} difficulties)')

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────────
# Expected time: ~12-15 min on T4, ~25-30 min on CPU
# Watch the APR and Goal% columns — these should increase over training

kt_model, agent, metrics = train(cfg, save_dir='checkpoints')

In [ ]:
# ── PLOT TRAINING CURVES ───────────────────────────────────────────────────

def smooth(x, w=50):
    return np.convolve(x, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# APR over episodes
axes[0].plot(smooth(metrics['aprs']), color='steelblue', linewidth=2)
axes[0].axhline(cfg.BETA, color='red', linestyle='--', alpha=0.7, label=f'Goal β={cfg.BETA}')
axes[0].set_title('Average Pass Rate over Training')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('APR')
axes[0].legend()
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)

# Reward over episodes
axes[1].plot(smooth(metrics['rewards']), color='darkorange', linewidth=2)
axes[1].set_title('Episode Reward over Training')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Total Reward')
axes[1].grid(alpha=0.3)

# Goal reached rate (rolling window)
goal_rate = smooth([float(g) for g in metrics['goals']])
axes[2].plot(goal_rate * 100, color='green', linewidth=2)
axes[2].set_title('Goal Reached % over Training')
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('% sessions reaching goal')
axes[2].set_ylim(0, 100)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=120)
plt.show()
print('Saved training_curves.png')

## Evaluation 1 — KT model quality
AUC > 0.70 = KT is reliably tracking student knowledge.
ECE near 0 = predicted probabilities are well-calibrated.

In [ ]:
kt_results = evaluate_kt(kt_model, cfg, n_students=300)

In [ ]:
# KT calibration curve and prediction histogram
labels = kt_results['labels']
preds  = kt_results['preds']

bins, bin_accs, bin_confs, bin_sizes = np.linspace(0,1,11), [], [], []
for i in range(10):
    lo, hi = bins[i], bins[i+1]
    mask   = (preds >= lo) & (preds < hi)
    if mask.sum() > 0:
        bin_accs.append(labels[mask].mean())
        bin_confs.append(preds[mask].mean())
        bin_sizes.append(mask.sum())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot([0,1],[0,1],'k--',alpha=0.5,label='Perfect calibration')
axes[0].scatter(bin_confs, bin_accs, s=[s/5 for s in bin_sizes],
                c=bin_confs, cmap='RdYlGn', vmin=0, vmax=1,
                edgecolors='gray', linewidth=0.5, zorder=3)
axes[0].set_xlabel('Mean predicted probability')
axes[0].set_ylabel('Fraction actually correct')
axes[0].set_title(f'KT Calibration Curve  (ECE={kt_results["ece"]:.3f})')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_xlim(0,1); axes[0].set_ylim(0,1)

axes[1].hist(preds[labels==1],bins=20,alpha=0.7,color='steelblue',label='Actually correct')
axes[1].hist(preds[labels==0],bins=20,alpha=0.7,color='tomato',   label='Actually wrong')
axes[1].axvline(0.5,color='black',linestyle='--',alpha=0.6)
axes[1].set_xlabel('Predicted mastery probability')
axes[1].set_ylabel('Count')
axes[1].set_title(f'KT Predictions  (AUC={kt_results["auc"]:.3f})')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('kt_evaluation.png', dpi=120)
plt.show()
print('Good result: blue bars right-heavy, red bars left-heavy')

## Evaluation 2 — Policy comparison
EPPO must beat Random clearly. Beating Greedy shows real learning beyond obvious heuristics.

In [ ]:
policy_results = evaluate_policy(kt_model, agent, cfg, n_students=300)

In [ ]:
# Policy comparison bar charts
policies = ['eppo', 'random', 'greedy']
colors   = ['steelblue', 'tomato', 'seagreen']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

aprs  = [policy_results[p]['mean_apr']   for p in policies]
stds  = [policy_results[p]['std_apr']    for p in policies]
goals = [policy_results[p]['goal_rate']  for p in policies]
steps = [policy_results[p]['mean_steps'] for p in policies]
sstds = [policy_results[p]['std_steps']  for p in policies]

axes[0].bar(policies, aprs, color=colors, alpha=0.85, yerr=stds, capsize=5)
axes[0].axhline(cfg.BETA,color='black',linestyle='--',alpha=0.5,label=f'beta={cfg.BETA}')
axes[0].set_title('Mean final APR'); axes[0].set_ylabel('APR')
axes[0].set_ylim(0,1); axes[0].legend(); axes[0].grid(axis='y',alpha=0.3)

axes[1].bar(policies, goals, color=colors, alpha=0.85)
axes[1].set_title('Goal reached %'); axes[1].set_ylabel('%')
axes[1].set_ylim(0,100); axes[1].grid(axis='y',alpha=0.3)

axes[2].bar(policies, steps, color=colors, alpha=0.85, yerr=sstds, capsize=5)
axes[2].set_title('Mean steps (lower = more efficient)')
axes[2].set_ylabel('Steps'); axes[2].grid(axis='y',alpha=0.3)

plt.suptitle('Policy Comparison: EPPO vs Baselines', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('policy_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Difficulty heatmap: what did EPPO learn to prefer per concept?
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, pname, color in zip(axes, policies, colors):
    dist     = policy_results[pname]['diff_dist']
    mastery  = policy_results[pname]['mean_mastery']
    norm     = mastery / mastery.max(axis=1, keepdims=True).clip(min=1e-6)
    im = ax.imshow(norm, aspect='auto', cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks([0,1,2]); ax.set_xticklabels(['Easy','Medium','Hard'])
    ax.set_yticks(range(cfg.N_CONCEPTS))
    ax.set_yticklabels(cfg.CONCEPTS, fontsize=8)
    ax.set_title(f'{pname.upper()}  (E:{dist[0]:.0%} M:{dist[1]:.0%} H:{dist[2]:.0%})')
    for i in range(cfg.N_CONCEPTS):
        for j in range(cfg.DIFF_LEVELS):
            ax.text(j,i,f'{mastery[i,j]:.2f}',ha='center',va='center',
                    fontsize=7, color='white' if norm[i,j]>0.6 else 'black')

plt.suptitle('Mean final mastery per concept per difficulty', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('mastery_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## Demo session
Watch EPPO make decisions step by step.

In [ ]:
run_demo_session(kt_model, agent, cfg)

## What to check before scaling up

Before moving to Phase 2 (dynamic concept set with sentence embeddings), verify:

1. **APR curve is trending up** — if it stays flat, something is broken in reward signal
2. **Goal% > 50%** after 1000 episodes — EPPO is learning a useful policy
3. **EPPO vs Random shows improvement** — EPPO must beat random meaningfully
4. **Mastery matrix updates after interactions** — KT model is learning
5. **Difficulty heatmap shows differentiation** — EPPO doesn't choose same difficulty for everything

## Phase 2 changes (adding sentence embeddings)

Only these two lines change in `SimpleKT.__init__`:

```python
# Phase 1 (current):
self.concept_embed = nn.Embedding(cfg.N_CONCEPTS, d)

# Phase 2 (dynamic concepts):
from sentence_transformers import SentenceTransformer
self.sentence_encoder = SentenceTransformer('all-MiniLM-L6-v2')
self.concept_proj = nn.Linear(384, d)  # trainable
```

In `forward`, replace:
```python
# Phase 1:
c = self.concept_embed(concept_ids)

# Phase 2:
raw = self.sentence_encoder.encode(concept_strings)  # strings, not IDs
c = self.concept_proj(torch.FloatTensor(raw))
```

Everything else — EPPO, reward function, training loop — is identical.